# 🫁 CheXpert Medical AI - Tự động Huấn luyện 1-Click trên Google Colab GPU

Notebook này đã được cấu hình tự động 100% với **Kaggle Token** và các tham số tối ưu (ConvNeXt-Small + Asymmetric Loss + Stanford U-Ones).

👉 **Cách dùng siêu đơn giản**: Bạn chỉ cần bấm menu **Runtime** $\rightarrow$ **Run all** (hoặc nhấn `Ctrl + F9`) và ngồi đợi kết quả!

## 1. Kiểm tra GPU Google Colab

In [ ]:
!nvidia-smi

## 2. Clone dự án & Cài đặt môi trường

In [ ]:
!git clone https://github.com/qdat2644/chex.git
%cd chex
!pip install -q -r requirements.txt
!pip install -q kaggle

## 3. Tự động xác thực Kaggle & Tải bộ dữ liệu CheXpert

In [ ]:
import os
import json

# Cấu hình tự động Kaggle Token
KAGGLE_TOKEN = "KGAT_66e946b810c28ffaeb913743194bd633"
os.environ["KAGGLE_API_TOKEN"] = KAGGLE_TOKEN
os.environ["KAGGLE_KEY"] = KAGGLE_TOKEN
os.environ["KAGGLE_USERNAME"] = "qdat2644"

kaggle_dir = os.path.expanduser("~/.kaggle")
os.makedirs(kaggle_dir, exist_ok=True)
kaggle_path = os.path.join(kaggle_dir, "kaggle.json")
with open(kaggle_path, "w") as f:
    json.dump({"username": "qdat2644", "key": KAGGLE_TOKEN, "token": KAGGLE_TOKEN}, f)
os.chmod(kaggle_path, 0o600)

print("Tự động tải bộ dữ liệu CheXpert từ Kaggle...")
!mkdir -p archive
!kaggle datasets download -d ashery/chexpert -p archive/ --unzip
print("Tải và giải nén dữ liệu hoàn tất!")

## 4. Huấn luyện Mô hình Tối ưu (ConvNeXt-Small + Asymmetric Loss + Stanford U-Ones)

In [ ]:
!python scripts/train.py \
    --data-root archive \
    --arch convnext_small \
    --loss asl \
    --image-size 224 \
    --epochs 6 \
    --batch-size 32 \
    --lr 1e-4 \
    --uncertain-policy u_ones_zeros \
    --scheduler cosine \
    --pretrained \
    --amp \
    --output checkpoints/chexpert_convnext_small.pt

## 5. Đánh giá & Tự động Hiệu chuẩn Ngưỡng phân loại F1 (Thresholds)

In [ ]:
!python scripts/evaluate.py \
    --data-root archive \
    --checkpoint checkpoints/chexpert_convnext_small.pt \
    --output-thresholds outputs/evaluation/thresholds.json

## 6. Tự động tải Checkpoint & Thresholds về máy

In [ ]:
from google.colab import files

print("Đang tải file model weights (.pt)...")
files.download('checkpoints/chexpert_convnext_small.pt')

print("Đang tải file thresholds.json...")
files.download('outputs/evaluation/thresholds.json')

print("Hoàn tất! Hãy copy 2 file này vào thư mục dự án tương ứng trên máy tính của bạn!")